# MCMC one parameter

In [21]:
import pymc3 as pm
import numpy as np
import theano.tensor as tt
import matplotlib.pyplot as plt
from ann_functions import import_data, normalization
from keras.models import load_model
from keras.initializers import glorot_uniform
import keras.backend as K
from keras.utils import custom_object_scope
import tensorflow as tf
import keras
import math

In [22]:
# reproducibility
seed = 42
np.random.seed(seed)
keras.utils.set_random_seed(seed)
tf.random.set_seed(seed)

# Data introduction

In [23]:
file_path_HF = "../DATA/reaction_diffusion_HF.mat"
(reaction_HF_test, U_HF_test) = import_data(file_path_HF)
U_HF_test = U_HF_test[:, -1, 44,44]

permutation = np.random.permutation(len(U_HF_test))
reaction_HF_test=normalization(reaction_HF_test)[permutation]
U_HF_test=normalization(U_HF_test)[permutation]


# personalized activation function
def custom_activation(x):
    
    return x + K.square(K.sin(x))

custom_objects = {'custom_activation': custom_activation }#, 'glorot_uniform': glorot_uniform()}

# Carica il modello utilizzando custom_objects
with custom_object_scope(custom_objects):

    keras_model1 = load_model("modelLF.h5")
    keras_model2 = load_model("modelLin.h5")
    keras_model3 = load_model("finalModel.h5")


def keras_predict(x):
    out1=keras_model1.predict(x)
    in2 = np.hstack((x, out1))
    out2=keras_model2.predict(in2)
    in3 = np.hstack((in2, out2))
    approx_model_output=keras_model3.predict(in3)
    
    return approx_model_output


In [24]:
n=1
reaction_HF=reaction_HF_test[0:n]
U_HF=U_HF_test[0:n]

def custom_operation(x, U_HF):
    return np.log(np.sqrt(2*np.pi)*np.sqrt(np.sqrt(np.mean(np.square(x - U_HF)))))*tt.sum((x - U_HF)**2)/(2*np.sqrt(np.mean(np.square(x - U_HF))))


with pm.Model() as model:
       # Definisci la variabile di input per il modello Keras
    input_var = pm.Data('input_var', reaction_HF)

    # Converti il tensor Theano in un array NumPy
    numpy_data = input_var.get_value().astype(np.float32)

    # Passa i dati a Keras e ottieni le previsioni
    keras_predictions = keras_predict(numpy_data)

    # Utilizza i dati Keras nelle operazioni Theano
    nn_potential = pm.Potential('nn_potential', custom_operation(keras_predictions, U_HF))

    # Definisci la likelihood (gestita dal potenziale)
    likelihood = pm.Normal('likelihood', mu=0.5, sd=0.5, observed=0)

    # Esegui l'inferenza MCMC
    trace = pm.sample(2000, tune=1000, cores=1, step=pm.Metropolis())
    # # input_var = pm.Data('input_var', reaction_HF)

    # # nn_potential = pm.Potential('nn_potential', custom_operation(keras_predict(input_var), U_HF))

    # # likelihood = pm.Normal('likelihood', mu=0.5, sd=0.5, observed=0)
    
    # # trace = pm.sample(2000, tune=1000, cores=1,step=pm.Metropolis())
    # Definisci la likelihood
 #   likelihood = pm.Normal('likelihood', mu=0, sd=1, observed=0)  # La likelihood è gestita dal potenziale

#     # Likelihood basato sui dati osservati
#     likelihood= pm.Normal('likelihood', mu=approx_model_output, sd=pm.HalfNormal('sigma', sd=1), observed=data_true)

#     # Likelihood basato sui dati corretti
#     #likelihood_true = pm.Normal('likelihood_true', mu=approx_model_output, sd=pm.HalfNormal('sigma', sd=1), observed=data_true)

#     # Campionamento MCMC
#     trace = pm.sample(2000, tune=1000, cores=1,step=pm.Metropolis())

# # Analisi dei risultati
print(pm.summary(trace).round(2))
pm.traceplot(trace)
# Estrai la traccia dei campioni per la variabile theta
theta_samples = trace['theta']

# Calcola la stima della media e la deviazione standard per ogni parametro
estimated_theta_mean = theta_samples.mean(axis=0)
estimated_theta_std = theta_samples.std(axis=0)

pm.plot_posterior(trace)
plt.show()

# Stampa le stime
print("Stime della media di theta:")
print(estimated_theta_mean)
print("\nDeviazione standard di theta:")
print(estimated_theta_std)

# Recupera le statistiche del sampler
#sampler_stats = trace.get_sampler_stats("metropolis")

# Estrai il numero di acceptance e rejection
acceptance_rates = sampler_stats[:, 0]  # Colonna 0: acceptance
rejection_rates = 1 - acceptance_rates
pm.autocorrplot(trace)

ppc = pm.sample_posterior_predictive(trace, samples=500, model=model)
# Plot acceptance e rejection
plt.plot(acceptance_rates, label="Acceptance Rate")
plt.plot(rejection_rates, label="Rejection Rate")
plt.xlabel("Iteration")
plt.ylabel("Rate")
plt.legend()
plt.show()

1/1 [==============================] - 0s 135ms/step


ValueError: No free random variables to sample.